In [42]:
import os
os.chdir("/home/jovyan/work/repo_conteo_gallinas")

import shutil
import glob
import json
import pandas as pd


In [43]:
from pathlib import Path

RUTA_DATA_PROCESSED = Path("data/processed")


## Lectura de información

In [23]:
lista_carpetas_eval = list()
for i in os.listdir(RUTA_DATA_PROCESSED):
    if os.path.isdir(RUTA_DATA_PROCESSED / i) and not i.startswith("."):
        for j in os.listdir(RUTA_DATA_PROCESSED / i):
            if j.endswith("json"):
                lista_carpetas_eval.append(str(RUTA_DATA_PROCESSED / i))

lista_carpetas_eval

['data/processed/20260728_070720_tp00009']

## Mover archivos

In [27]:
# JSON
for i in lista_carpetas_eval:
   for j in os.listdir(i):
       if j.endswith("json"):
           ruta_destino = f"data/deteccion/labels/{j}"
           ruta_origen = f"{i}/{j}"
           shutil.copy(ruta_origen, ruta_destino)


In [28]:
# IMGS
for i in lista_carpetas_eval:
   for j in os.listdir(i):
       if j.endswith("jpg"):
           ruta_destino = f"data/deteccion/imgs/{j}"
           ruta_origen = f"{i}/{j}"
           shutil.copy(ruta_origen, ruta_destino)


## Estructura

In [32]:
download_imgs_path = "data/deteccion/imgs"
download_labels_path = "data/deteccion/labels"


all_path_imgs = glob.glob(os.path.join(download_imgs_path, "*.jpg"))
all_paths_json = glob.glob(os.path.join(download_labels_path, "*.json"))


In [38]:
# Leer los archivos json y juntarlos todos en una colección
all_data = {}
for file_path in all_paths_json:
    name_file = os.path.basename(file_path)
    with open(file_path, "r") as file:
        data = json.load(file)
    all_data[name_file] = data


In [44]:
data = []

for via_project, one_json in all_data.items():
    for _, value_dict in one_json["_via_img_metadata"].items():

        name_file = value_dict["filename"]

        for region in value_dict["regions"]:

            shape_attrs = region["shape_attributes"]
            region_attrs = region["region_attributes"]

            if shape_attrs["name"] == "rect":

                x = shape_attrs["x"]
                y = shape_attrs["y"]
                w = shape_attrs["width"]
                h = shape_attrs["height"]

                segmentation = [
                    x, y,
                    x + w, y,
                    x + w, y + h,
                    x, y + h
                ]

            elif shape_attrs["name"] == "polygon":

                segmentation = []

                for px, py in zip(
                    shape_attrs["all_points_x"],
                    shape_attrs["all_points_y"]
                ):
                    segmentation.extend([px, py])

            else:
                continue

            data.append({
                "via_project": via_project,
                "filename": name_file,
                "objeto": region_attrs.get("object", "gallina"),
                "segmentation": segmentation
            })

df = pd.DataFrame(data)
df


,via_project,filename,objeto,segmentation
0,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t0000s_1.jpg,gallina,"[563, 677, 630, 677, 630, 774, 563, 774]"
1,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t0000s_1.jpg,gallina,"[694, 792, 879, 792, 879, 902, 694, 902]"
2,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t0000s_1.jpg,gallina,"[1724, 670, 1879, 670, 1879, 852, 1724, 852]"
3,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t0000s_1.jpg,gallina,"[1684, 1012, 1834, 1012, 1834, 1070, 1684, 1070]"
4,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t0000s_1.jpg,gallina,"[1359, 982, 1503, 982, 1503, 1068, 1359, 1068]"
...,...,...,...,...
6865,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t1240s_63.jpg,gallina,"[859, 30, 941, 30, 941, 129, 859, 129]"
6866,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t1240s_63.jpg,gallina,"[743, 6, 853, 6, 853, 85, 743, 85]"
6867,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t1240s_63.jpg,gallina,"[13, 533, 69, 533, 69, 554, 13, 554]"
6868,via_project_13Oct2026_7h57mBRIGIT (1).json,20260728_070720_tp00009_t1240s_63.jpg,gallina,"[9, 565, 67, 565, 67, 606, 9, 606]"


In [45]:
# Juntar el nombre del archivo con su ubicación
filename_to_filepath = {os.path.basename(path): path for path in all_path_imgs}

# Agregar la columna al datafram
df['filepath'] = df['filename'].map(filename_to_filepath)

In [47]:
from siuba import *
df = df >> filter(-_.objeto.isna())


In [48]:
# Obtener los nombres de las imágenes
unique_files = df["filename"].unique()


In [49]:
len(unique_files)

62

In [53]:
class_mapping = {
    "gallina": 0
}


In [54]:
def segmentation_to_yolo_bbox(segmentation,
                              image_width,
                              image_height):

    xs = segmentation[::2]
    ys = segmentation[1::2]

    x_min = min(xs)
    x_max = max(xs)

    y_min = min(ys)
    y_max = max(ys)

    x_center = ((x_min + x_max) / 2) / image_width
    y_center = ((y_min + y_max) / 2) / image_height

    width = (x_max - x_min) / image_width
    height = (y_max - y_min) / image_height

    return x_center, y_center, width, height



In [55]:
import cv2
import os

def export_labels_yolo_detection(
    image_path,
    segmentations,
    objects,
    output_folder
):

    image = cv2.imread(image_path)

    h, w = image.shape[:2]

    base_name = os.path.splitext(
        os.path.basename(image_path)
    )[0]

    label_path = os.path.join(
        output_folder,
        f"{base_name}.txt"
    )

    with open(label_path, "w") as file:

        for seg, obj in zip(segmentations, objects):

            cls_id = class_mapping[obj]

            xc, yc, bw, bh = segmentation_to_yolo_bbox(
                seg,
                w,
                h
            )

            file.write(
                f"{cls_id} {xc} {yc} {bw} {bh}\n"
            )



## Partición de la data

In [56]:
from sklearn.model_selection import train_test_split

unique_files = df["filename"].unique()

train_imgs, val_imgs = train_test_split(
    unique_files,
    test_size=0.2,
    random_state=123
)

train_df = (
    df[df["filename"].isin(train_imgs)]
    .reset_index(drop=True)
)

val_df = (
    df[df["filename"].isin(val_imgs)]
    .reset_index(drop=True)
)

print(train_df.shape)
print(val_df.shape)


(5374, 5)
(1496, 5)


In [60]:
len(train_df["filename"].unique().tolist()), len(val_df["filename"].unique().tolist())



(49, 13)

## Preparación de la data para el entrenamiento

In [61]:
import shutil

def process_split(
    split_df,
    target_image_dir,
    target_label_dir
):

    os.makedirs(target_image_dir, exist_ok=True)
    os.makedirs(target_label_dir, exist_ok=True)

    for filename in split_df["filename"].unique():

        instances = (
            split_df[
                split_df["filename"] == filename
            ]
        )

        image_path = instances["filepath"].iloc[0]

        shutil.copy(
            image_path,
            os.path.join(
                target_image_dir,
                os.path.basename(image_path)
            )
        )

        export_labels_yolo_detection(
            image_path=image_path,
            segmentations=instances["segmentation"].tolist(),
            objects=instances["objeto"].tolist(),
            output_folder=target_label_dir
        )


In [62]:
process_split(
    train_df,
    "data/dataset/train/images",
    "data/dataset/train/labels"
)

process_split(
    val_df,
    "data/dataset/val/images",
    "data/dataset/val/labels"
)
